# 08 · Benchmark: SkewerJAX against fake_spectra

The setup:
- **Inputs:** both codes compute Lyα and Si III λ1206 on identical skewers of the same TNG50 snapshots (snapshot 25,
  z = 3), at 1 km/s pixels. fake_spectra runs with its default (top-hat) kernel.
- **Process isolation:** every configuration ran in its own process, pinned to a fixed CPU set with a fixed thread
  count (`OMP_NUM_THREADS` for fake_spectra's OpenMP, the same CPU affinity for XLA).
- **Timing:** each process ran several repeats; the figures show medians with min–max bars. The wall time is split
  into setup, I/O, cell selection, gas physics and τ.
- **Agreement:** SkewerJAX's matched mode against fake_spectra's τ on the first skewers, and the mean flux of
  SkewerJAX's default mode.

This notebook only reads the saved results (`RESULTS_DIR`); the runs come from
`scripts/benchmarks/benchmark_vs_fake_spectra.py`. Runtime: seconds.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import (use_style, ENTITY_COLORS as EC, MODE_STYLES as MS, TOL_BRIGHT as TOL, LABELS, FIGSIZES,
                                      add_panel_letters, plot_pipeline, plot_spectra, zoom_inset)
use_style()
%matplotlib inline
import json
from skewerjax.comparison.benchmark import load_results, summary, STAGES
from skewerjax.utils.plotting import (plot_bench_scaling, plot_bench_stages, plot_bench_memory, plot_bench_speedup, plot_bench_accuracy,
                                      plot_bench_resolution, bench_select, bench_key, BENCH_STYLES)
RESULTS_DIR = os.environ.get("SKEWERJAX_BENCHMARK_RESULTS", "../claude_agents/tmp_files/20261021_2330/scratch/benchmarks")
R = load_results(RESULTS_DIR)
ok = [r for r in R if r.get("status") == "ok"]
print(f"{len(R)} results in {RESULTS_DIR}: {len(ok)} ok, {len(R) - len(ok)} timed out")
m = ok[0]["metadata"]
print(f"CPU: {m['cpu']} ({m['logical_cpus']} logical CPUs), {m['ram_GB']:.0f} GB RAM | GPU: {m['gpu']}")
print("software:", {k: v for k, v in m["versions"].items() if v}, "| Python", m["python"])

In [ ]:
plot_pipeline(('simulation', 'tau', 'comparison'), note="filled: the steps of this notebook"); plt.show()

## Summary table

Median wall time [s] per stage (all repeats), JIT compilation of SkewerJAX's τ, peak host memory, and the matched-mode flux agreement (max |ΔF| on the first skewers, Lyα / Si III).

In [ ]:
rows = []
for r in sorted(ok, key=lambda r: (r["config"]["snap_base"], r["config"]["n_skewers"], r["config"]["threads"], str(bench_key(r)))):
    s, c, a = summary(r), r["config"], r.get("agreement") or {}
    rows.append([c["snap_base"].split("/")[-1], c["n_skewers"], c["threads"], BENCH_STYLES[bench_key(r)]["label"], s["n_repeats"],
                 *[f"{s[k][0]:.2f}" for k in STAGES], f"{s['total'][0]:.1f}", "" if r.get("compile_s") is None else f"{r['compile_s']:.1f}",
                 f"{r['peak_rss_GB']:.2f}", "/".join(f"{a[l]['max_dF']:.0e}" for l in ("HI-LyA", "SiIII-1206")) if a else ""])
head = ["sim", "N_sk", "thr", "code", "rep", *STAGES, "total", "compile", "RSS GB", "max|dF|"]
w = [max(len(str(x)) for x in col) for col in zip(head, *rows)]
for row in [head] + rows:
    print("  ".join(str(x).rjust(n) for x, n in zip(row, w)))
for r in R:
    if r.get("status") != "ok":
        print(f"not measured: {r['id']} ({r['status']} after {r['wall_s']:.0f} s)")

## 1 · Time against the number of skewers (TNG50-4, all 32 threads)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=FIGSIZES["wide"], layout="constrained")
plot_bench_scaling(axs[0], ok, sim="TNG50-4", threads=32, what="total"); axs[0].set_title("total")
plot_bench_scaling(axs[1], ok, sim="TNG50-4", threads=32, what="tau"); axs[1].set_title(r"$\tau$ only (first repeat includes the JIT compilation)")
axs[0].legend(fontsize=12); add_panel_letters(axs); plt.show()

*Takeaway:* from 1 to 256 skewers, fake_spectra goes from 1.6 s to 6.5 s and SkewerJAX (CPU) from 5.3 s to 447 s. SkewerJAX's τ costs about 1.6 s per skewer (two lines, 3961 pixels), fake_spectra's about 0.02 s, so the gap widens from 3.4× to 69×.

## 2 · Where the time goes (TNG50-4, 64 skewers)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=FIGSIZES["wide"], layout="constrained")
plot_bench_stages(axs[0], ok, sim="TNG50-4", n_skewers=64, threads=32)
plot_bench_stages(axs[1], ok, sim="TNG50-4", n_skewers=64, threads=32, fraction=True); axs[1].get_legend().remove()
add_panel_letters(axs); plt.show()

*Takeaway:* at 64 skewers, τ is 91 % of SkewerJAX's CPU time (103 of 114 s), and the cell selection (5.9 s against 0.08 s) is the second gap. SkewerJAX's I/O is the shorter one (0.31 s against 0.96 s).

## 3 · Memory

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=FIGSIZES["wide"], layout="constrained")
plot_bench_memory(axs[0], ok, sim="TNG50-4", threads=32); plot_bench_memory(axs[1], ok, sim="TNG50-4", threads=32, device=True)
axs[0].legend(fontsize=12); axs[1].set_title("GPU (XLA_PYTHON_CLIENT_PREALLOCATE=false)"); add_panel_letters(axs); plt.show()

*Takeaway:* fake_spectra peaks at 0.5 GB of host memory for every skewer count. SkewerJAX grows by about 60 MB per skewer (1.5 GB at 1 skewer, 5.1 GB at 64, 16 GB at 256), and the GPU peak grows from 0.14 GB to 3.5 GB.

## 4 · Speed-up, threads and resolution

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=FIGSIZES["triple"], layout="constrained")
plot_bench_speedup(axs[0], ok, sim="TNG50-4", threads=32, what="total")
plot_bench_scaling(axs[1], ok, x="threads", sim="TNG50-4", n_skewers=16, platform="cpu", dtype=None); axs[1].set_title("16 skewers, CPU")
plot_bench_resolution(axs[2], ok, n_skewers=16)
axs[0].legend(fontsize=11); axs[1].legend(fontsize=11); add_panel_letters(axs); plt.show()

*Takeaway:* no configuration beats fake_spectra.
- **Float32** cuts SkewerJAX's CPU τ by 2.5× (16 skewers).
- **Threads:** SkewerJAX's τ scales 7.8× from 1 to 8 performance cores, and then only 1.17× more up to 32 threads.
- **GPU:** the RTX 3060 is 1.8× slower than the CPU in float64, and 3.8× slower in float32.
- **Resolution:** the ratio falls to 1.5× at TNG50-2 resolution, where fake_spectra's reading of the 349 GB snapshot dominates and SkewerJAX's cell selection takes 248 s.

## 5 · Accuracy: did both codes compute the same thing?

Filled: SkewerJAX's matched mode against fake_spectra (max |ΔF|, first 4 skewers, float64). Open: the mean-flux difference of SkewerJAX's default mode (all skewers), i.e. the size of the modelling differences that notebook 06 breaks down. Dotted: 5e-5.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=FIGSIZES["wide"], layout="constrained", sharey=True)
plot_bench_accuracy(axs[0], ok, line="HI-LyA"); plot_bench_accuracy(axs[1], ok, line="SiIII-1206")
axs[0].set_title(r"Ly$\alpha$"); axs[1].set_title("Si III 1206"); axs[1].set_ylabel("")
for ax in axs: ax.axhline(5e-5, color="k", ls=":", lw=1)
axs[0].set_ylim(top=0.5); axs[0].legend(fontsize=12, loc="lower right")
add_panel_letters(axs); plt.show()

*Takeaway:* in matched mode, the two codes agree to max |ΔF| ≤ 2.3e-5 (Lyα) and 2.2e-7 (Si III) on TNG50-4. On TNG50-3 the figures are 2.8e-5 and 6.2e-5, and on TNG50-2 6.4e-5 and 1.2e-3. The TNG50-2 Si III value is above the dotted 5e-5 line, and its cause has not been investigated. SkewerJAX's default mode gives a lower Lyα mean flux (0.500 against 0.533 at 16 skewers), which notebook 06 breaks down effect by effect.

## Summary

- **Agreement:** on identical skewers the two codes compute the same spectra; the matched mode agrees to |ΔF| ~ 1e-5.
- **Speed:** fake_spectra is faster on this machine in every configuration measured, from 1.5× to 69×. The gap is
  SkewerJAX's τ kernel (an exact Voigt profile for every absorber at every pixel) and, at high resolution, its cell
  selection.
- **Options:** float32 (2.5× on the τ) and more CPU performance cores help SkewerJAX. This GPU is slower than
  the CPU in both precisions.
- **Reruns:** `scripts/benchmarks/benchmark_vs_fake_spectra.py --sets quick,skewers,resolution` repeats the
  measurements and skips the configurations already saved; set `SKEWERJAX_BENCHMARK_RESULTS` to read another folder
  here.